# Introduction

This notebook aims to understand and visualize real-time GTFS (RT) data reported from the TTC.
Specifically we would like to visualize Route 29, a bus route that is known to have significantly high trip times.

We would like to compare the state of the transit network (with respect to Route 29) to the scheduled state (through GTFS static data provided
by the TTC)

You can find the exact data source used in this project below: 
GTFS Static data: https://open.toronto.ca/dataset/merged-gtfs-ttc-routes-and-schedules/
GTFS RT data: https://gtfsrt.ttc.ca/

# Data Modeling

Before we dig into the data itself, we will start off by understanding the inherent structure of the data (the data model). The GTFS  specifciation is a global standard used to structure transit data reported by transit agencies. Specifically the GTFS specifies an array of CSV text files each representing different types of transit information (i.e. route/trip information, schedule information, geo-shape information etc.). Each CSV text file relate to each other through a relational schema.

A relational UML schema representing the GTFS RT data reported by the TTC can be found below:

![GTFS_RealTime_Relational_Diagram](GTFS_RT_Relational_Diagram.jpeg)


It is likely that GTFS static data is needed to provide complementary information to the GTFS RT data, so the relational schema for the GTFS static data is also provided below:

![GTFS_STATIC_Relational_Model](GTFS_STATIC_Relational_Model.jpeg)

# Data Collection and Preprocessing

GTFS-RT data provides insight into the instantaneous state of the transit system. 

For example, trip_update provides stoptime event data from the current vehicle position/stop onwards and veichle_position measures the instantaneous speed, location and position of a veichle going through a trip/route.

Therefore in order to have sufficient data for analysis, we will need to query TTC's RT GTFS api multiple times to get historical data; and then calculate some running average to evalaute the state of the transit system.

To implement this, we will need to break the problem down into 2 steps:

a. create a data collection and preprocessing script that continously collects, formats and processes queried data and saves it into a unified database 

b. create a data analysis script that analyzes the recorded data in the database

## Data Collection

We will start off by performing a. 

Real-time data was collected using GET requests to the TTC GTFS-RT api

In [1]:
import requests
import subprocess
import json
import pandas as pd
import re
from datetime import datetime

alerts_url = "https://gtfsrt.ttc.ca/alerts/all?format=text"
trip_updates_url = "https://gtfsrt.ttc.ca/trips/update?format=text"
veichle_positions_url = "https://gtfsrt.ttc.ca/vehicles/position?format=text"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
}

/Users/devrajsolanki/Documents/TAL/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def collect_data(trip_updates_url, veichle_positions_url):
    # reading trip updates data
    trip_updates_response = requests.get(trip_updates_url, headers=headers)
    trip_updates_text = trip_updates_response.text

    # reading vehicle position data
    veichle_positions_response = requests.get(veichle_positions_url, headers=headers)
    veichle_positions_text = veichle_positions_response.text
    return trip_updates_text, veichle_positions_text


In [3]:
# Additonally static data will be loaded 

import pandas as pd

static_schedule_trip_info = pd.read_csv("trip_metrics.csv")


/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_28750/896794257.py:5: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  static_schedule_trip_info = pd.read_csv("trip_metrics.csv")


## Data Formatting

We will format the queried data according to a modified schema based on the GTFS RT schema diagram

![Modified_GTFS_RT_Relational_Diagram.jpeg](Modified_GTFS_RT_Relational_Diagram.jpeg)

In [4]:
# made with claude code

# takes in raw trip_update text data and formats it into a dataframe according to the modified GTFS RT schema above
def modify_trip_updates(response_text: str) -> list[dict]:
    text = response_text

    # gets all information in each "block" of information w/r to each entity
    def blocks(src, key):
        for m in re.finditer(rf'\b{key}\s*\{{', src):
            depth, i = 0, m.end() - 1
            while i < len(src):
                depth += (src[i] == '{') - (src[i] == '}')
                if depth == 0: yield src[m.end():i]; break
                i += 1

    # some blocks dont repeat i.e. veichle id or trip information w/r to each entity, so we just get the first match or return none
    def first(src, key):
        return next(blocks(src, key), None)

    # given a block of text, we return the value in the key:value pairs stored within the feed data (i.e. stop_sequence: val )
    def val(src, key):
        m = re.search(rf'\b{key}\s*:\s*"?([^"\n]+?)"?\s*$', src or '', re.M)
        return m.group(1) if m else None

    # get timestamp of feed
    feed_timestamp = val(first(text, 'header'), 'timestamp')

    # record each stop update within a trip update as one dicionary or record in the TripUpdate dataframe
    records = []
    for entity in blocks(text, 'entity'):
        tu = first(entity, 'trip_update')
        if not tu: continue
        trip = first(tu, 'trip')
        veh  = first(tu, 'vehicle')
        base = {
            'id':                         val(entity, 'id'),
            'trip_id':                    val(trip,   'trip_id'),
            'trip_schedule_relationship': val(trip,   'schedule_relationship'),
            'route_id':                   val(trip,   'route_id'),
            'vehicle_id':                 val(veh,    'id'),
            'timestamp':                  val(tu,     'timestamp'),
        }
        for stu in blocks(tu, 'stop_time_update'):
            records.append({**base,
                'stop_sequence':              val(stu, 'stop_sequence'),
                'stop_id':                    val(stu, 'stop_id'),
                'stop_schedule_relationship': val(stu, 'schedule_relationship'),
                'arrival_time':               val(first(stu, 'arrival'),   'time'),
                'departure_time':             val(first(stu, 'departure'), 'time'),
            })
    return pd.DataFrame(records), feed_timestamp



In [5]:
# made with claude code

# takes in raw vehicle_positions text data and formats it into a dataframe according to the modified GTFS RT schema above

def modify_vehicle_positions(response_text: str) -> list[dict]:
    text = response_text
 
    def blocks(src, key):
        for m in re.finditer(rf'\b{key}\s*\{{', src):
            depth, i = 0, m.end() - 1
            while i < len(src):
                depth += (src[i] == '{') - (src[i] == '}')
                if depth == 0: yield src[m.end():i]; break
                i += 1
 
    def first(src, key):
        return next(blocks(src, key), None)
 
    def val(src, key):
        m = re.search(rf'\b{key}\s*:\s*"?([^"\n]+?)"?\s*$', src or '', re.M)
        return m.group(1) if m else None
 
    feed_timestamp = val(first(text, 'header'), 'timestamp')

    records = []
    for entity in blocks(text, 'entity'):
        vp = first(entity, 'vehicle')
        if not vp: continue
        trip = first(vp, 'trip')
        pos  = first(vp, 'position')
        veh  = first(vp, 'vehicle')      # nested vehicle { id: ... }
        records.append({
            'id':                    val(entity, 'id'),
            'latitude':              val(pos,    'latitude'),
            'longitude':             val(pos,    'longitude'),
            'bearing':               val(pos,    'bearing'),
            'speed':                 val(pos,    'speed'),
            'timestamp':             val(vp,     'timestamp'),
            'vehicle_id':            val(veh,    'id'),
            'occupancy_status':      val(vp,     'occupancy_status'),
            'current_stop_sequence': val(vp,     'current_stop_sequence'),
            'current_status':        val(vp,     'current_status'),
            'stop_id':               val(vp,     'stop_id'),
            'trip_id':               val(trip,   'trip_id'),
            'schedule_relationship': val(trip,   'schedule_relationship'),
            'route_id':              val(trip,   'route_id'),
        })
    return pd.DataFrame(records), feed_timestamp


## Data Processing

In [6]:
# processing for trip_updates

def process_trip_updates(df):
    df = df.copy()
    
    # replace No_Data schedule relationships with NaN for arrival/departure times
    df.loc[df['stop_schedule_relationship'] == 'NO_DATA', ['arrival_time', 'departure_time']] = pd.NA

    # combine arrival_time and departure_time into a single column
    df['arrival/departure_time'] = df['arrival_time'].combine_first(df['departure_time'])

    # convert arrival/departure time from POSIX to datetime (UTC -> local)
    df['arrival/departure_time'] = pd.to_datetime(df['arrival/departure_time'], unit='s', errors='coerce').dt.tz_localize('UTC').dt.tz_convert('America/Toronto').dt.tz_localize(None)

    # drop arrival and departure time columns
    df = df.drop(columns=['arrival_time', 'departure_time'])
    
    # replace missing vehicle_id with NaN
    df['vehicle_id'] = df['vehicle_id'].replace('', pd.NA)

    # convert timestamp from POSIX to datetime (UTC -> local)
    df['timestamp'] = pd.to_datetime(df['timestamp'].astype(int), unit='s').dt.tz_localize('UTC').dt.tz_convert('America/Toronto').dt.tz_localize(None)

    return df.reset_index(drop=True)


In [7]:
# preprocessing for vehicle_positions

def process_vehicle_positions(df):
    df = df.copy()
    
    # impute missing occupancy_status with most common value
    df['occupancy_status'] = df['occupancy_status'].fillna(df['occupancy_status'].mode()[0])

    # drop rows where route_id is missing (when route_id is missing, so is trip_id, trip_schedule_relationship etc. [trip_id, stop_id, status])
    df = df.dropna(subset=['route_id'])

    # convert speed from m/s to km/hr
    df['speed_km/hr'] = df['speed'].astype(float) * 3.6

    # change timestamp from POSIX to current time
    df['timestamp'] = pd.to_datetime(df['timestamp'].astype(int), unit='s').dt.tz_localize('UTC').dt.tz_convert('America/Toronto').dt.tz_localize(None)

    return df.reset_index(drop=True)



In [8]:
# save data to an sqlite database so data can be recovered regardless of notebook session

import sqlite3

def save_to_sqlite(trip_updates_dict, vehicle_positions_dict, db_path='gtfs_rt.db'):
    with sqlite3.connect(db_path) as conn:
        for timestamp, df in trip_updates_dict.items():
            df['feed_timestamp'] = timestamp
            df.to_sql('trip_updates', conn, if_exists='append', index=False)

        for timestamp, df in vehicle_positions_dict.items():
            df['feed_timestamp'] = timestamp
            df.to_sql('vehicle_positions', conn, if_exists='append', index=False)




In [9]:
# query data, format it and save the timestamped data for safekeeping
trip_updates_txt, veichle_positions_txt = collect_data(trip_updates_url, veichle_positions_url)

trip_updates, tu_feed_time = modify_trip_updates(trip_updates_txt)
vehicle_positions, vp_feed_time  = modify_vehicle_positions(veichle_positions_txt)

trip_updates = process_trip_updates(trip_updates)
vehicle_positions = process_vehicle_positions(vehicle_positions)


trip_updates_dict = {datetime.fromtimestamp(int(tu_feed_time)): trip_updates}
vehicle_positions_dict = {datetime.fromtimestamp(int(vp_feed_time)): vehicle_positions}

save_to_sqlite(trip_updates_dict, vehicle_positions_dict)

/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_28750/4254561454.py:13: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['arrival/departure_time'] = pd.to_datetime(df['arrival/departure_time'], unit='s', errors='coerce').dt.tz_localize('UTC').dt.tz_convert('America/Toronto').dt.tz_localize(None)


# Data Analysis

In [10]:
# open current data
import sqlite3
import pandas as pd

with sqlite3.connect('gtfs_rt.db') as conn:
    trip_updates = pd.read_sql('SELECT * FROM trip_updates', conn)
    vehicle_positions = pd.read_sql('SELECT * FROM vehicle_positions', conn)

# reformat time columns to datetime types
trip_updates['arrival/departure_time'] = pd.to_datetime(trip_updates['arrival/departure_time'])
trip_updates['timestamp'] = pd.to_datetime(trip_updates['timestamp'])
trip_updates['feed_timestamp'] = pd.to_datetime(trip_updates['feed_timestamp'])
vehicle_positions['timestamp'] = pd.to_datetime(vehicle_positions['timestamp'])
vehicle_positions['feed_timestamp'] = pd.to_datetime(vehicle_positions['feed_timestamp'])

# reformat location data to numeric types
vehicle_positions['latitude'] = pd.to_numeric(vehicle_positions['latitude'])
vehicle_positions['longitude'] = pd.to_numeric(vehicle_positions['longitude'])

In [11]:
# join trip_update with static schedule data to get the scheduled context for trips as well as route information
import duckdb

trip_updates_with_static_schedule = duckdb.query("""
SELECT
    tu.*,
    tsi.arrival_time_hhmmss AS scheduled_arrival_time,
    tsi.departure_time_hhmmss AS scheduled_departure_time,
    tsi."Speed_Km/hr_between_stops" AS scheduled_between_stop_speed_km_hr,
    tsi."Average_trip_speed_km/hr" AS scheduled_trip_speed_km_hr,
    tsi.direction_of_travel_binary AS direction_of_travel_binary,
    tsi.direction_of_travel_name AS direction_of_travel_name,
    tsi.stop_name AS stop_name,
    tsi.stop_code AS stop_code,
    tsi.stop_lat as stop_lat,
    tsi.stop_lon as stop_lon,
    tsi.route_short_name AS route_short_name,
    tsi.route_long_name AS route_long_name,
    tsi.trip_start_time AS scheduled_trip_start_time,
    tsi.trip_end_time AS scheduled_trip_end_time,
    tsi.binary_service_monday AS scheduled_binary_service_monday,
    tsi.binary_service_tuesday AS scheduled_binary_service_tuesday,
    tsi.binary_service_wednesday AS scheduled_binary_service_wednesday,
    tsi.binary_service_thursday AS scheduled_binary_service_thursday,
    tsi.binary_service_friday AS scheduled_binary_service_friday,
    tsi.binary_service_saturday AS scheduled_binary_service_saturday,
    tsi.binary_service_sunday AS scheduled_binary_service_sunday
                              
FROM trip_updates tu
LEFT JOIN static_schedule_trip_info tsi ON tu.trip_id = tsi.trip_id AND tu.stop_sequence = tsi.stop_sequence
""")

# convert to dataframe
trip_updates_with_static_schedule = trip_updates_with_static_schedule.to_df()

# reformat time columns to datetime types
trip_updates_with_static_schedule['arrival/departure_time'] = pd.to_datetime(trip_updates_with_static_schedule['arrival/departure_time'])
trip_updates_with_static_schedule['timestamp'] = pd.to_datetime(trip_updates_with_static_schedule['timestamp'])
trip_updates_with_static_schedule['feed_timestamp'] = pd.to_datetime(trip_updates_with_static_schedule['feed_timestamp'])

# include today's date along with scheduled time
trip_updates_with_static_schedule['scheduled_arrival_datetime'] = trip_updates_with_static_schedule.apply(lambda row: pd.Timestamp.combine(pd.Timestamp.today().date(), pd.to_datetime(row['scheduled_arrival_time'], format='%H:%M:%S').time()) if pd.notna(row['scheduled_arrival_time']) else pd.NaT, axis=1)
trip_updates_with_static_schedule['scheduled_departure_datetime'] = trip_updates_with_static_schedule.apply(lambda row: pd.Timestamp.combine(pd.Timestamp.today().date(), pd.to_datetime(row['scheduled_departure_time'], format='%H:%M:%S').time()) if pd.notna(row['scheduled_departure_time']) else pd.NaT, axis=1)
trip_updates_with_static_schedule['scheduled_trip_start_datetime'] = trip_updates_with_static_schedule.apply(lambda row: pd.Timestamp.combine(pd.Timestamp.today().date(), pd.to_datetime(row['scheduled_trip_start_time'], format='%H:%M:%S').time()) if pd.notna(row['scheduled_trip_start_time']) else pd.NaT, axis=1)   
trip_updates_with_static_schedule['scheduled_trip_end_datetime'] = trip_updates_with_static_schedule.apply(lambda row: pd.Timestamp.combine(pd.Timestamp.today().date(), pd.to_datetime(row['scheduled_trip_end_time'], format='%H:%M:%S').time()) if pd.notna(row['scheduled_trip_end_time']) else pd.NaT, axis=1) 

#### Current Snapshot of transit system for Route 29

In [12]:
# get current feed data by filtering for the most recent feed_timestamp
recent_vehicle_positions = vehicle_positions[vehicle_positions["feed_timestamp"] == vehicle_positions["feed_timestamp"].max()]
recent_trip_updates = trip_updates[trip_updates["feed_timestamp"] == trip_updates["feed_timestamp"].max()]
recent_trip_updates_with_static_schedule = trip_updates_with_static_schedule[trip_updates_with_static_schedule["feed_timestamp"] == trip_updates_with_static_schedule["feed_timestamp"].max()]

In [13]:
# count of unique trips currently servicing route 29 using vehicle_positions [most recent feed_timestamp]
recent_route_29_trips = recent_vehicle_positions[recent_vehicle_positions['route_id'] == '29']['trip_id'].nunique()
print(f"Number of unique trips currently servicing route 29 according to vehicle_positions: {recent_route_29_trips}")


Number of unique trips currently servicing route 29 according to vehicle_positions: 15


In [14]:
# count of unique trips that are planned to service route 29 according to trip_updates
route_29_planned_trips = recent_trip_updates[recent_trip_updates['route_id'] == '29']['trip_id'].nunique()

# also get the time range of the trip updates feed for context (using arrival/departure time)
feed_time_range_start = recent_trip_updates['arrival/departure_time'].min()
feed_time_range_end = recent_trip_updates['arrival/departure_time'].max()
print(f"Number of unique trips currently planned to service route 29 between {feed_time_range_start} and {feed_time_range_end}: {route_29_planned_trips}")

Number of unique trips currently planned to service route 29 between 2026-06-01 19:23:32 and 2026-06-01 20:33:15: 21


In [15]:
# get average speed of vehicles currentlyservicing route 29 according to vehicle_positions
route_29_speeds = recent_vehicle_positions[recent_vehicle_positions['route_id'] == '29']['speed_km/hr'].dropna()
average_speed_route_29 = route_29_speeds.mean()
print(f"Average speed of vehicles currently servicing route 29: {average_speed_route_29:.2f} km/hr")

Average speed of vehicles currently servicing route 29: 17.81 km/hr


In [16]:
# get distribution of occupancy status of vehicles currently servicing route 29 according to vehicle_positions
route_29_occupancy = recent_vehicle_positions[recent_vehicle_positions['route_id'] == '29']['occupancy_status'].dropna()
occupancy_distribution_route_29 = route_29_occupancy.value_counts()
print(f"Occupancy status distribution of vehicles currently servicing route 29:")
print(occupancy_distribution_route_29)

Occupancy status distribution of vehicles currently servicing route 29:
occupancy_status
EMPTY                  10
FEW_SEATS_AVAILABLE     5
Name: count, dtype: int64


In [17]:
# calculate the delay of each stop in recent_trip_updates_with_static_schedule by comparing the arrival/departure_time with scheduled_arrival_time

recent_trip_updates_with_static_schedule['delay'] = (recent_trip_updates_with_static_schedule['arrival/departure_time'] - recent_trip_updates_with_static_schedule['scheduled_arrival_datetime']).dt.total_seconds()

# look at distribution of delays for route 29
route_29_delays = recent_trip_updates_with_static_schedule[recent_trip_updates_with_static_schedule['route_id'] == '29']['delay'].dropna()
print(f" Current Delay distribution across all stops for all trips for route 29 (in seconds):")
print(route_29_delays.describe())

 Current Delay distribution across all stops for all trips for route 29 (in seconds):
count    349.000000
mean     -36.644699
std      197.236336
min     -474.000000
25%     -122.000000
50%        0.000000
75%        0.000000
max      458.000000
Name: delay, dtype: float64


/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_28750/1605553128.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recent_trip_updates_with_static_schedule['delay'] = (recent_trip_updates_with_static_schedule['arrival/departure_time'] - recent_trip_updates_with_static_schedule['scheduled_arrival_datetime']).dt.total_seconds()


In [18]:
# calculate the delay per stop for route 29 to see if some stops are more delayed/less delayed than others on average, include stop code and name

current_route_29_stop_delays = recent_trip_updates_with_static_schedule[recent_trip_updates_with_static_schedule['route_id'] == '29'].groupby(['stop_id', 'stop_name', 'stop_code'])['delay'].mean().reset_index().sort_values(by='delay', ascending=True)

# get top 3 most delayed stops on average for route 29
most_delayed_stops_route_29 = current_route_29_stop_delays.tail(3)
print(f"Top 3 currently most delayed stops on average for route 29:")
display(most_delayed_stops_route_29)

# get top 3 least delayed stops on average for route 29
least_delayed_stops_route_29 = current_route_29_stop_delays.head(3)
print(f"Top 3 currently least delayed stops on average for route 29:")
display(least_delayed_stops_route_29)

Top 3 currently most delayed stops on average for route 29:


,stop_id,stop_name,stop_code,delay
37,4379,Dufferin St at Jane Osler Blvd - Yorkdale Shop...,985,105.0
36,4378,Dufferin St at Yorkdale Rd,979,105.0
66,8048,Dufferin St at Ranee Ave North Side,982,153.4


Top 3 currently least delayed stops on average for route 29:


,stop_id,stop_name,stop_code,delay
83,9497,Dufferin St at Ascot Ave,2017,-204.000000
48,6283,Dufferin St at Hope St,2061,-204.000000
44,5487,Dufferin St at Shanly St,2089,-181.666667


In [19]:
# visualize the delay of stop geographically on a map to see if there are any spatial patterns in the delays for route 29
# made with claude code

route_29_stop_delays = recent_trip_updates_with_static_schedule[recent_trip_updates_with_static_schedule['route_id'] == '29'].groupby(['stop_id', 'stop_name', 'stop_code', 'stop_lat', 'stop_lon'])['delay'].mean().reset_index()

import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

map_center_lat = route_29_stop_delays['stop_lat'].mean()
map_center_lon = route_29_stop_delays['stop_lon'].mean()
route_29_map = folium.Map(location=[map_center_lat, map_center_lon], zoom_start=12, tiles='CartoDB dark_matter')

# center the colormap around 0 (on time) so green=early, red=late
delays = route_29_stop_delays['delay']
abs_max = max(abs(delays.min()), abs(delays.max()))
norm = mcolors.TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
colormap = plt.cm.RdYlGn_r  # green=early, red=late

for _, row in route_29_stop_delays.iterrows():
    color = mcolors.to_hex(colormap(norm(row['delay'])))
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=7, color=color, weight=2,
        fill=True, fill_color=color, fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>{row['stop_name']}</b><br>"
            f"Avg Delay: {row['delay']/60:.1f} min",
            max_width=220
        ),
        tooltip=f"{row['stop_name']}: {row['delay']/60:.1f} min"
    ).add_to(route_29_map)

legend_html = """
<div style="position:fixed;bottom:40px;left:40px;background:rgba(20,20,20,0.85);border:1px solid #555;border-radius:8px;padding:12px 16px;z-index:9999;font-family:monospace;color:#eee;font-size:13px;">
  <b>Avg Delay</b><br><br>
  <span style="color:#1a9850;">&#9632;</span> Early<br>
  <span style="color:#fee08b;">&#9632;</span> On time<br>
  <span style="color:#d73027;">&#9632;</span> Late
</div>
"""

route_29_map.get_root().html.add_child(folium.Element(legend_html))
route_29_map.save("route_29_current_stop_delays_map.html")

In [20]:
# visualize current occupancy and speed of vehicles currently servicing route 29 geographically 

recent_vehicle_positions_route_29 = recent_vehicle_positions[recent_vehicle_positions['route_id'] == '29']

import folium

map_center_lat = recent_vehicle_positions_route_29['latitude'].mean()
map_center_lon = recent_vehicle_positions_route_29['longitude'].mean()
route_29_vehicle_map = folium.Map(location=[map_center_lat, map_center_lon], zoom_start=12, tiles='CartoDB dark_matter')

for _, row in recent_vehicle_positions_route_29.iterrows():
    if row['occupancy_status'] == 'EMPTY':
        color = 'green'
    elif row['occupancy_status'] == 'MANY_SEATS_AVAILABLE':
        color = 'lightgreen'
    elif row['occupancy_status'] == 'FEW_SEATS_AVAILABLE':
        color = 'orange'
    elif row['occupancy_status'] == 'FULL':
        color = 'red'
    else:
        color = 'gray'

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6, color=color, weight=1,
        fill=True, fill_color=color, fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>Vehicle ID: {row['vehicle_id']}</b><br>"
            f"Speed: {row['speed_km/hr']:.1f} km/hr<br>"
            f"Occupancy: {row['occupancy_status'].replace('_', ' ').title()}<br>"
            f"Bearing: {row['bearing']} degrees",
            max_width=250
        ),
        tooltip=f"Vehicle ID: {row['vehicle_id']} - Speed: {row['speed_km/hr']:.1f} km/hr - Occupancy: {row['occupancy_status'].replace('_', ' ').title()}"
    ).add_to(route_29_vehicle_map)

vehicle_legend_html = """
<div style="position:fixed;bottom:40px;right:40px;background:rgba(20,20,20,0.85);border:1px solid #555;border-radius:8px;padding:12px 16px;z-index:9999;font-family:monospace;color:#eee;font-size:13px;">
  <b>Vehicle Occupancy</b><br><br>
  <span style="color:green;">&#9632;</span> Empty<br>
  <span style="color:lightgreen;">&#9632;</span> Many Seats Available<br>
  <span style="color:orange;">&#9632;</span> Few Seats Available<br>
  <span style="color:red;">&#9632;</span> Full<br>
  <span style="color:gray;">&#9632;</span> Unknown
</div>
"""
route_29_vehicle_map.get_root().html.add_child(folium.Element(vehicle_legend_html))
route_29_vehicle_map.save("route_29_current_vehicle_positions_map.html")

In [28]:
# calculate current headway between veichles currently servicing route 29 [using recent_trip_updates]

recent_route_29_data = recent_trip_updates_with_static_schedule[
    recent_trip_updates_with_static_schedule['route_id'] == '29'
].copy()

# sometimes we have added trips that are not scheduled so we need to identify the direction of travel for these trips
# we will do this by matching their planned sequence of stops to the set of stops known to be in the Northbound or Southbound direction of travel
    # set of stops in northbound direction of travel for route 29 according to static schedule data

northbound_stops_route_29 = set(
    static_schedule_trip_info[
        (static_schedule_trip_info['route_short_name'] == '29') &
        (static_schedule_trip_info['direction_of_travel_binary'] == 1)
    ]['stop_id'].unique()
)
southbound_stops_route_29 = set(
    static_schedule_trip_info[
        (static_schedule_trip_info['route_short_name'] == '29') &
        (static_schedule_trip_info['direction_of_travel_binary'] == 0)
    ]['stop_id'].unique()
)

    # if trip_schedule_relationship = "ADDED", match sequence of stop_ids for that trip to the set of stop_ids in northbound vs southbound direction to infer direction of travel for that trip
def infer_direction_of_travel(row):
    if row['trip_schedule_relationship'] == 'ADDED':
        trip_stop_ids = set(
            recent_trip_updates_with_static_schedule[
                recent_trip_updates_with_static_schedule['trip_id'] == row['trip_id']
            ]['stop_id']
        )
        return 1 if len(trip_stop_ids.intersection(northbound_stops_route_29)) > len(trip_stop_ids.intersection(southbound_stops_route_29)) else 0
    return row['direction_of_travel_binary']

# additionally, impute missing stop_code, stop_name, stop_lat, stop_lon for trips with trip_schedule_relationship = "ADDED" using the stop_id 
# create a dictionary with stop_id as key and stop_name, stop_code, stop_lat, stop_lon as values for quick lookup: using recent_route_29_data

stop_info_dict = recent_route_29_data.dropna(subset=['stop_id']).drop_duplicates(subset=['stop_id']).set_index('stop_id')[['stop_name', 'stop_code', 'stop_lat', 'stop_lon']].to_dict(orient='index')

def impute_stop_info(row):
    if row['trip_schedule_relationship'] == 'ADDED':
        stop_info = stop_info_dict.get(row['stop_id'], {})
        row['stop_name'] = stop_info.get('stop_name', row['stop_name'])
        row['stop_code'] = stop_info.get('stop_code', row['stop_code'])
        row['stop_lat'] = stop_info.get('stop_lat', row['stop_lat'])
        row['stop_lon'] = stop_info.get('stop_lon', row['stop_lon'])
    return row


recent_route_29_data['direction_of_travel_binary'] = recent_route_29_data.apply(infer_direction_of_travel, axis=1)
recent_route_29_data = recent_route_29_data.apply(impute_stop_info, axis=1)

# headway = time difference between consecutive vehicles servicing the same stop
# for each stop code and direction of travel order all stop departure/arrival times from earliest to latest and calculate the time difference between each successive trip arrival time
recent_route_29_data = recent_route_29_data.sort_values(
    ['stop_id', 'direction_of_travel_binary', 'arrival/departure_time']
).reset_index(drop=True)

# groupby stop code and direction of travel, calculate headway as the time difference between consecutive stop arrivals for each stop and direction of travel
recent_route_29_data['headway_seconds'] = recent_route_29_data.groupby(['stop_id', 'direction_of_travel_binary'])['arrival/departure_time'].diff().dt.total_seconds()

In [29]:
# calculate average headway across all stops for route 29

average_headway_route_29 = recent_route_29_data['headway_seconds'].mean()
print(f"Average headway across all stops for route 29: {average_headway_route_29:.1f} seconds")

Average headway across all stops for route 29: 437.6 seconds


In [30]:
# visualize the average headway per stop for route 29 geographically on a map
# made with claude code

route_29_stop_headways = recent_route_29_data.groupby(['stop_id', 'stop_name', 'stop_code', 'stop_lat', 'stop_lon'])['headway_seconds'].mean().reset_index()

map_center_lat = route_29_stop_headways['stop_lat'].mean()
map_center_lon = route_29_stop_headways['stop_lon'].mean()
route_29_headway_map = folium.Map(location=[map_center_lat, map_center_lon], zoom_start=12, tiles='CartoDB dark_matter')

# continuous color scale — green = short headway, red = long headway, gray = unknown
valid_headways = route_29_stop_headways['headway_seconds'].dropna()
norm = mcolors.Normalize(vmin=valid_headways.min(), vmax=valid_headways.max())
colormap = plt.cm.RdYlGn_r

for _, row in route_29_stop_headways.iterrows():
    if pd.isna(row['headway_seconds']):
        color = 'gray'
        headway_text = 'Unknown'
    else:
        color = mcolors.to_hex(colormap(norm(row['headway_seconds'])))
        headway_text = f"{row['headway_seconds']/60:.1f} min"

    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=7, color=color, weight=2,
        fill=True, fill_color=color, fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>{row['stop_name']}</b><br>"
            f"Avg Headway: {headway_text}",
            max_width=220
        ),
        tooltip=f"{row['stop_name']}: {headway_text}"
    ).add_to(route_29_headway_map)

headway_legend_html = """
<div style="position:fixed;bottom:40px;left:40px;background:rgba(20,20,20,0.85);border:1px solid #555;border-radius:8px;padding:12px 16px;z-index:9999;font-family:monospace;color:#eee;font-size:13px;">
  <b>Avg Headway</b><br><br>
  <span style="color:#1a9850;">&#9632;</span> Short wait<br>
  <span style="color:#fee08b;">&#9632;</span> Moderate wait<br>
  <span style="color:#d73027;">&#9632;</span> Long wait<br>
  <span style="color:gray;">&#9632;</span> Unknown
</div>
"""
route_29_headway_map.get_root().html.add_child(folium.Element(headway_legend_html))
route_29_headway_map.save("route_29_current_headways_map.html")

